# 1a. Shape, dtype, rank

About 20 minutes.

By the end: you can look at a tensor and say its shape, its rank, and how
many numbers it holds. You can also tell a scalar from a length-1 vector,
which matters more than it sounds.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False, linewidth=110)


class _Todo:
    def __repr__(self):
        return "<fill me in>"


TODO = _Todo()


def S(t):
    """Shape as a tuple. Type this constantly."""
    return tuple(t.shape)


def check(label, got, want=None, shape=None, dtype=None, atol=1e-5, rtol=1e-4):
    """Check one answer. Prints what went wrong, then raises."""
    bad = []
    if isinstance(got, _Todo):
        bad.append("not filled in yet")
    else:
        if shape is not None:
            if not torch.is_tensor(got):
                bad.append(f"expected a tensor, got {type(got).__name__}")
            elif S(got) != tuple(shape):
                bad.append(f"shape is {S(got)}, want {tuple(shape)}")
        if dtype is not None and torch.is_tensor(got) and got.dtype != dtype:
            bad.append(f"dtype is {got.dtype}, want {dtype}")
        if want is not None:
            g = got if torch.is_tensor(got) else torch.as_tensor(got)
            w = want if torch.is_tensor(want) else torch.as_tensor(want)
            if S(g) != S(w):
                bad.append(f"shape is {S(g)}, want {S(w)}")
            elif not torch.allclose(g.double(), w.double(), atol=atol, rtol=rtol):
                bad.append(f"values differ, largest gap "
                           f"{(g.double() - w.double()).abs().max().item():.3e}")
    if bad:
        print(f"FAIL  {label}")
        for b in bad:
            print(f"      {b}")
        raise AssertionError(label)
    print(f"ok    {label}")


def same(label, a, b):
    """Two shapes or tuples must match."""
    if isinstance(a, _Todo):
        print(f"FAIL  {label}: not filled in yet")
        raise AssertionError(label)
    if tuple(a) != tuple(b):
        print(f"FAIL  {label}\n      you said {tuple(a)}, it is {tuple(b)}")
        raise AssertionError(label)
    print(f"ok    {label} = {tuple(a)}")

## The three sizes

In [ ]:
x = torch.randn(2, 3, 4)

print("shape", S(x))
print("ndim ", x.ndim)      # how many axes. the rank.
print("numel", x.numel())   # total numbers. the product of the shape.

Axis `0` is outermost. Axis `-1` is innermost, and its elements sit next to
each other in memory.

In this project `-1` is nearly always the axis you want: the hidden axis, the
head axis, the vocabulary axis.

## dtype

A tensor holds one kind of number. Mixing them is a common bug.

In [ ]:
print(torch.randn(3).dtype)                      # float32, the default
print(torch.tensor([7, 2]).dtype)                # int64 from python ints
print((torch.tensor([7, 2]) > 3).dtype)          # bool from a comparison
print(torch.zeros(3, dtype=torch.uint8).dtype)   # what packed weights arrive as

## Drill 1 (easy)

Make these three, using `torch.zeros`:

- `a` with 24 elements and rank 3
- `b` holding bytes
- `c` holding booleans

In [ ]:
a = TODO
b = TODO
c = TODO

check("a has 24 elements", torch.tensor(a.numel()), want=24)
check("a is rank 3", torch.tensor(a.ndim), want=3)
check("b is bytes", b, dtype=torch.uint8)
check("c is bools", c, dtype=torch.bool)

## Rank 0 is a real thing

A full reduction gives shape `()`. Not a Python float. Not a length-1 vector.

In [ ]:
t = torch.randn(4)
s = t.sum()

print("t.sum()       ", S(s), " rank", s.ndim)
print("t.sum().item()", type(s.item()).__name__)
print("t.sum()[None] ", S(s[None]), " rank", s[None].ndim)

All three are "the same number". Only the rank-0 one works in an `if`. Only
`.item()` gives you something you can put in an f-string without ceremony.

## Drill 2 (medium)

From `logits` of shape `(T, V)`, produce exactly these ranks:

- `best` — the winning index per row, shape `(T,)`
- `best_col` — the same thing as a column, shape `(T, 1)`
- `top` — each row's largest value, shape `(T, 1)`
- `peak` — the single largest value anywhere, as a Python float

In [ ]:
T, V = 3, 10
logits = torch.randn(T, V)

best = TODO
best_col = TODO
top = TODO
peak = TODO

check("best", best, shape=(T,))
check("best_col", best_col, shape=(T, 1))
check("they agree", best_col[:, 0], want=best)
check("top", top, shape=(T, 1))
check("top lines up with logits", (logits - top).amax(-1), want=torch.zeros(T))
assert isinstance(peak, float), f"peak is {type(peak).__name__}, want float"
print("ok    peak is a python float")

`top` has shape `(T, 1)` so that `logits - top` works. That is what the
`keepdim` argument is for, and notebook **1g** is about nothing else.

## Next

**1b** — pulling pieces out of a tensor.